In [2]:
# no use of DynaSent-R1, DynaSent-R2, or SST-3
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load pre-trained tokenizer and model
model_name = "finiteautomata/bertweet-base-sentiment-analysis"
#model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
model

C:\Users\sgrebenkin\.conda\envs\cs224u\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (position_embeddings): Embedding(130, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

# Test

In [3]:
import torch.nn.functional as F
text = "The movie was fantastic!"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
with torch.no_grad():
    outputs = model(**inputs)
logits = outputs.logits
probs = F.softmax(logits, dim=-1)
predicted_class = torch.argmax(probs, dim=-1).item()
predicted_class

2

# Dataset Preparation

In [4]:
#import pandas as pd
#df = pd.read_csv("hf://datasets/EnZon3/The-Worlds-Sentiment/dataset.csv")

from datasets import load_dataset

dataset = load_dataset("EnZon3/The-Worlds-Sentiment")
dataset

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['Headline', 'Sentiment'],
        num_rows: 3089
    })
})

In [21]:
dataset = load_dataset("EnZon3/The-Worlds-Sentiment")

dataset = dataset.rename_column("Sentiment", "score")
dataset = dataset.rename_column("Headline", "sentence")
dataset = dataset.select_columns(['sentence', 'score'])

# Tokenize dataset
def preprocess_sentence(examples):
    return tokenizer(examples['sentence'], padding="max_length", truncation=True)

dataset['train'] = dataset['train'].add_column('labels', [0 if i < 0 else 2 if i > 0 else 1 for i in dataset['train']['score']])

tokenized_dataset = dataset.map(preprocess_sentence, batched=True)
tokenized_dataset = tokenized_dataset['train'].train_test_split(test_size=0.3)

train_dataset = tokenized_dataset['train'].shuffle()
test_dataset = tokenized_dataset['test'].shuffle()

Repo card metadata block was not found. Setting CardData to empty.


Map:   0%|          | 0/3089 [00:00<?, ? examples/s]

In [22]:
train_dataset

Dataset({
    features: ['sentence', 'score', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2162
})

In [38]:
# Freeze all parameters in the model first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the RobertaIntermediate, RobertaOutput, and RobertaClassificationHead layers
for layer in model.roberta.encoder.layer:
    layer.intermediate.requires_grad = True  # Unfreeze RobertaIntermediate layers
    layer.output.requires_grad = True       # Unfreeze RobertaOutput layers

# Unfreeze the classification head
for param in model.classifier.parameters():
    param.requires_grad = True

# Print out which layers are trainable for verification
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"Trainable layer: {name}")

Trainable layer: classifier.dense.weight
Trainable layer: classifier.dense.bias
Trainable layer: classifier.out_proj.weight
Trainable layer: classifier.out_proj.bias


In [39]:
from transformers import Trainer, TrainingArguments


# Set up training arguments and Trainer
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    learning_rate=2e-6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

# Fine-tune the model
trainer.train()

# Evaluate the model
results = trainer.evaluate()
print(results)

Epoch,Training Loss,Validation Loss
1,No log,1.527793
2,0.205600,1.508080
3,0.205600,1.491496
4,0.213300,1.488332
5,0.213300,1.485415
6,0.177900,1.486510
7,0.177900,1.483531
8,0.191300,1.480964
9,0.191300,1.480875
10,0.177700,1.480042


{'eval_loss': 1.4800423383712769, 'eval_runtime': 1.7867, 'eval_samples_per_second': 518.833, 'eval_steps_per_second': 64.924, 'epoch': 10.0}


# Bakeoff

In [14]:
import os
import wget

if not os.path.exists(os.path.join("data", "sentiment", "cs224u-sentiment-test-unlabeled.csv")):
    os.makedirs(os.path.join('data', 'sentiment'), exist_ok=True)
    wget.download('https://web.stanford.edu/class/cs224u/data/cs224u-sentiment-test-unlabeled.csv', out='data/sentiment/')

100% [............................................................................] 275836 / 275836

In [16]:
import pandas as pd

bakeoff_df = pd.read_csv(
    os.path.join("data", "sentiment", "cs224u-sentiment-test-unlabeled.csv"))

In [52]:
bakeoff_df

,example_id,sentence
0,0,This year we were at a restaurant that clearly...
1,1,A long way.
2,2,A friend and I went on a Thursday evening aro...
3,3,You'll love to say I used to be married to tha...
4,4,I feel like any place I move will be a downgra...
...,...,...
2995,2995,despite its many infuriating flaws -- not the ...
2996,2996,A bone cyst is a hollow spot of bone filled wi...
2997,2997,The portions are big & the check is small.
2998,2998,Service and food was mediocre at best.
